In [4]:
import pandas as pd
import pyarrow .parquet as pq
trips=pq.read_table("uber_hourly_full.parquet")


In [5]:
df=trips.to_pandas()
df.shape

(1522152, 9)

In [6]:
H=1
d=df.sort_values(["zone_id","hour"]).reset_index(drop=True)
assert d.groupby("zone_id").size().nunique()==1 ,"every zone schould have every hour"
import numpy as np
d.loc[d["avg_wait_sec"]<0,"avg_wait_sec"]=np.nan
d["service_zone"]=d["service_zone"].fillna("Unknown")
d["Borough"]=d["Borough"].fillna("Unknown")
from pandas.tseries.holiday import USFederalHolidayCalendar

d["HourofDay"]=d["hour"].dt.hour
d["dayofweek"]=d["hour"].dt.dayofweek
d["isweekend"]=(d["dayofweek"]>=5).astype(int)
Holidays=USFederalHolidayCalendar().holidays("2026-01-01","2026-08-31")
d["is_airport"]=d["service_zone"].isin(["Airports","EWR"]).astype(int)
d["isholiday"]=d["hour"].dt.normalize().isin(Holidays).astype(int)
g=d.groupby("zone_id")["trips"]
lags=sorted({L for L in (H,H+1,H+2,24,48,168)if L >=H})
for L in lags:
  d[f"lag_{L}"]=g.shift(L)
d["roll24"]=g.transform(lambda s:s.shift(H).rolling(24).mean())
d["roll168"]=g.transform(lambda s:s.shift(H).rolling(168).mean())
d["waitlag"]=d.groupby("zone_id")["avg_wait_sec"].shift(H)
d["zone_cat"]=d["zone_id"].astype("category")
d["borough_cat"]=d["Borough"].astype("category")
d=d.dropna(subset=["lag_168","roll168"]).reset_index(drop=True)
train = d[d["hour"] <  "2026-07-01"]
valid = d[(d["hour"] >= "2026-07-01") & (d["hour"] < "2026-08-01")]
test  = d[d["hour"] >= "2026-08-01"]        # locked: do not touch until the very end
print("train", train.shape, "| valid", valid.shape, "| test", test.shape)

train (1089936, 25) | valid (194184, 25) | test (194184, 25)


In [7]:
train=d[d["hour"]<"2026-07-01"]

In [8]:
def score(y,p):
  y,p=np.asarray(y,float),np.asarray(p,float)
  return {
      "MAE":np.abs(y-p).mean(),
      "RMSE":np.sqrt(((y-p)**2).mean()),
      "WAPE%":100*np.abs(y-p).sum()/y.sum()}

results={}
results["lastvalue"]=score(valid["trips"],valid[f"lag_{H}"])
results["same hour yesterday"]=score(valid["trips"],valid["lag_24"])
results["same hour last week"]=score(valid["trips"],valid["lag_168"])
print(pd.DataFrame(results).T.round(2))

                       MAE   RMSE  WAPE%
lastvalue            16.08  27.43  20.53
same hour yesterday  18.83  38.03  24.04
same hour last week  15.30  29.50  19.52


In [9]:
import lightgbm as lgb
feats=(["HourofDay","dayofweek","isweekend","is_airport","isholiday","waitlag","zone_cat","borough_cat","roll24","roll168"] + [f"lag_{L}" for L in lags])
params=dict(objective="poisson",learning_rate=0.05,num_leaves=63,min_data_in_leaf=100,feature_fraction=0.8,bagging_fraction=0.8,bagging_freq=1,seed=42,verbose=-1)


In [ ]:
dtrain=lgb.Dataset(train[feats],train["trips"])
dvalid=lgb.Dataset(valid[feats],valid["trips"],reference=dtrain)
Model=lgb.train(params,dtrain,num_boost_round=2000,valid_sets=[dvalid],callbacks=[lgb.early_stopping(50),lgb.log_evaluation(100)])

Training until validation scores don't improve for 50 rounds
[100]	valid_0's poisson: -297.904


In [ ]:
pred = Model.predict(valid[feats], num_iteration=Model.best_iteration)
results["LightGBM (Poisson)"] = score(valid["trips"], pred)
print(pd.DataFrame(results).T.round(2))

In [ ]:
top20=train.groupby("zone_id")["trips"].sum().nlargest(20).index
v=valid.assign(pred=pred)
segments={"all zones":v["trips"]>=0,
          "airports":v["is_airport"]==1,
          "top 20 zones":v["zone_id"].isin(top20),
          "all other zones":~v["zone_id"].isin(top20),
          "holidays":v["isholiday"]==1
          }

rows={}
for name,m in segments.items():
  s=v[m]
  rows[name]={"Model Wape %":score(s["trips"],s["pred"])["WAPE%"],
              "last_week wape%":score(s["trips"],s["lag_168"])["WAPE%"]}
print(pd.DataFrame(rows).T.round(2))

In [ ]:
results_h24  = pd.DataFrame(results).T
segments_h24 = pd.DataFrame(rows).T

In [ ]:
#cell4
import matplotlib.pyplot as plt

v = valid.assign(pred=pred)
v["abs_err"] = (v["trips"] - v["pred"]).abs()

def wape_by(col):
    g = v.groupby(col)[["abs_err", "trips"]].sum()
    return (100 * g["abs_err"] / g["trips"]).round(1)

print("WAPE % by hour of day:\n", wape_by("HourofDay").to_string())
print("\nWAPE % by day of week (0 = Monday):\n", wape_by("dayofweek").to_string())

# zones that contribute the most error, and their own WAPE
z = v.groupby(["zone_id", "Zone"])[["abs_err", "trips"]].sum()
z["WAPE_%"] = (100 * z["abs_err"] / z["trips"]).round(1)
z["share_of_total_error_%"] = (100 * z["abs_err"] / z["abs_err"].sum()).round(2)
print("\nTop 10 zones by share of total error:\n",
      z.sort_values("share_of_total_error_%", ascending=False).head(10))

# one picture: actual vs forecast for the busiest zone, one week of July
zid = train.groupby("zone_id")["trips"].sum().idxmax()
s = v[(v["zone_id"] == zid) & (v["hour"] >= "2026-07-06") & (v["hour"] < "2026-07-13")]
plt.figure(figsize=(12, 3.5))
plt.plot(s["hour"], s["trips"], label="actual")
plt.plot(s["hour"], s["pred"], label="forecast")
plt.title(f"Zone {zid}: actual vs forecast, 6-12 July (horizon = {H} h)")
plt.legend(); plt.tight_layout(); plt.show()

In [ ]:
#cell5
# 80% prediction interval: a 10% model and a 90% model
base = dict(learning_rate=0.1, num_leaves=63, min_data_in_leaf=100, feature_fraction=0.8,
            bagging_fraction=0.8, bagging_freq=1, seed=42, verbose=-1)
dtrain = lgb.Dataset(train[feats], train["trips"])
dvalid = lgb.Dataset(valid[feats], valid["trips"], reference=dtrain)

bounds = {}
for name, alpha in {"lo": 0.1, "hi": 0.9}.items():
    m = lgb.train({**base, "objective": "quantile", "alpha": alpha}, dtrain,
                  num_boost_round=500, valid_sets=[dvalid],
                  callbacks=[lgb.early_stopping(50), lgb.log_evaluation(100)])
    bounds[name] = m.predict(valid[feats], num_iteration=m.best_iteration)

lo = np.clip(np.minimum(bounds["lo"], bounds["hi"]), 0, None)
hi = np.maximum(bounds["lo"], bounds["hi"])
v = valid.assign(pred=pred, lo=lo, hi=hi)
v["inside"] = (v["trips"] >= v["lo"]) & (v["trips"] <= v["hi"])

print(f"Overall coverage: {100 * v['inside'].mean():.1f}%  (target: 80%)")
print(f"Average interval width: {(v['hi'] - v['lo']).mean():.1f} trips "
      f"(average demand per zone-hour: {v['trips'].mean():.1f})")

top20 = train.groupby("zone_id")["trips"].sum().nlargest(20).index
segs = {"airports": v["is_airport"] == 1,
        "top-20 zones": v["zone_id"].isin(top20),
        "all other zones": ~v["zone_id"].isin(top20),
        "holidays": v["isholiday"] == 1}
print(pd.Series({k: round(100 * v.loc[m, "inside"].mean(), 1) for k, m in segs.items()},
                name="coverage_%").to_string())

In [ ]:
# (a) which days are hardest? (does 3-5 July stand out?)
w = valid.assign(pred=pred)
w["abs_err"] = (w["trips"] - w["pred"]).abs()
daily = w.groupby(w["hour"].dt.date)[["abs_err", "trips"]].sum()
print("WAPE % by day:\n", (100 * daily["abs_err"] / daily["trips"]).round(1).to_string())

# (b) widen the intervals using the first half of July, then check on the second half
v = valid.assign(pred=pred, lo=lo, hi=hi).reset_index(drop=True)
cal = v[v["hour"] <  "2026-07-16"]        # used to calibrate
chk = v[v["hour"] >= "2026-07-16"]        # only used to check

# bigger zone-hours need bigger adjustments, so calibrate separately by expected volume
edges = np.quantile(cal["pred"], [0.2, 0.4, 0.6, 0.8])
grp = lambda p: np.digitize(p, edges)     # 0 = quietest ... 4 = busiest

adj = np.zeros(5)
miss = np.maximum(cal["lo"] - cal["trips"], cal["trips"] - cal["hi"])   # how far outside; negative = inside
for gi in range(5):
    s = miss[grp(cal["pred"]) == gi]
    n = len(s)
    adj[gi] = np.quantile(s, min(1.0, np.ceil((n + 1) * 0.8) / n))
print("adjustment per volume group (trips):", adj.round(1))

a = adj[grp(chk["pred"])]
chk = chk.assign(lo2=np.clip(chk["lo"] - a, 0, None), hi2=chk["hi"] + a)
chk["hi2"] = np.maximum(chk["hi2"], chk["lo2"])

top20 = train.groupby("zone_id")["trips"].sum().nlargest(20).index
segs = {"all zones": chk["trips"] >= 0,
        "airports": chk["is_airport"] == 1,
        "top-20 zones": chk["zone_id"].isin(top20),
        "all other zones": ~chk["zone_id"].isin(top20)}
before = (chk["trips"] >= chk["lo"])  & (chk["trips"] <= chk["hi"])
after  = (chk["trips"] >= chk["lo2"]) & (chk["trips"] <= chk["hi2"])
print(pd.DataFrame({"coverage before %": {k: 100 * before[m].mean() for k, m in segs.items()},
                    "coverage after %":  {k: 100 * after[m].mean()  for k, m in segs.items()}}).round(1))
print("average width before:", round((chk["hi"] - chk["lo"]).mean(), 1),
      "| after:", round((chk["hi2"] - chk["lo2"]).mean(), 1))

In [ ]:
import requests

# 1. Hourly NYC weather from Open-Meteo (ERA5 reanalysis; free, no key for non-commercial use).
#    unixtime avoids the DST problem: the API ignores DST changes when you ask for a named timezone.
r = requests.get("https://archive-api.open-meteo.com/v1/archive", params={
    "latitude": 40.7128, "longitude": -74.0060,
    "start_date": "2025-12-31", "end_date": "2026-09-01",
    "hourly": "temperature_2m,precipitation,snowfall,wind_speed_10m",
    "timeformat": "unixtime"}, timeout=120)
r.raise_for_status()
print("units:", r.json()["hourly_units"])

weather = pd.DataFrame(r.json()["hourly"])
weather["hour"] = (pd.to_datetime(weather["time"], unit="s", utc=True)
                   .dt.tz_convert("America/New_York").dt.tz_localize(None))   # NYC local clock, like the TLC data
weather = weather.drop(columns="time").rename(columns={
    "temperature_2m": "temp_c", "precipitation": "precip_mm",
    "snowfall": "snow_cm", "wind_speed_10m": "wind_kmh"})
wcols = ["temp_c", "precip_mm", "snow_cm", "wind_kmh"]

# 2. Attach weather to the existing splits (weather of the hour we predict is a forecast input, so no shifting)
train_w = train.merge(weather, on="hour", how="left")
valid_w = valid.merge(weather, on="hour", how="left")
print("July hours without weather:", valid_w["temp_c"].isna().mean())

# 3. Same settings for both models, so the comparison is fair
fast = dict(objective="poisson", learning_rate=0.1, num_leaves=63, min_data_in_leaf=100,
            feature_fraction=0.8, bagging_fraction=0.8, bagging_freq=1, seed=42, verbose=-1)

def fit_predict(cols):
    dtr = lgb.Dataset(train_w[cols], train_w["trips"])
    dva = lgb.Dataset(valid_w[cols], valid_w["trips"], reference=dtr)
    m = lgb.train(fast, dtr, num_boost_round=500, valid_sets=[dva],
                  callbacks=[lgb.early_stopping(50), lgb.log_evaluation(0)])
    return m.predict(valid_w[cols], num_iteration=m.best_iteration)

p_without = fit_predict(feats)
p_with    = fit_predict(feats + wcols)

def wape(y, p):
    y, p = np.asarray(y, float), np.asarray(p, float)
    return 100 * np.abs(y - p).sum() / y.sum()

segs = {"all hours": valid_w["trips"] >= 0,
        "airports": valid_w["is_airport"] == 1,
        "weekends": valid_w["dayofweek"] >= 5,
        "rainy hours (> 0.2 mm)": valid_w["precip_mm"] > 0.2,
        "dry hours": valid_w["precip_mm"] <= 0.2}
out = {}
for name, m in segs.items():
    y = valid_w.loc[m, "trips"]
    out[name] = {"WAPE without weather %": wape(y, p_without[m.values]),
                 "WAPE with weather %":    wape(y, p_with[m.values]),
                 "share of hours %": 100 * m.mean()}
print(pd.DataFrame(out).T.round(2).to_string())

In [ ]:
import lightgbm as lgb

def wape(y, p):
    y, p = np.asarray(y, float), np.asarray(p, float)
    return 100 * np.abs(y - p).sum() / y.sum()

def metrics(y, p):
    y, p = np.asarray(y, float), np.asarray(p, float)
    return {"MAE": np.abs(y - p).mean(), "RMSE": np.sqrt(((y - p) ** 2).mean()), "WAPE_%": wape(y, p)}

feats = (["HourofDay", "dayofweek", "isweekend", "isholiday", "is_airport",
          "zone_cat", "borough_cat", "waitlag", "roll24", "roll168"]
         + [f"lag_{L}" for L in lags])

# ---- 1. point model: trained on Jan-Jun, early stopping on July (same settings as validated) ----
point_params = dict(objective="poisson", learning_rate=0.05, num_leaves=63, min_data_in_leaf=100,
                    feature_fraction=0.8, bagging_fraction=0.8, bagging_freq=1, seed=42, verbose=-1)
dtr = lgb.Dataset(train[feats], train["trips"])
dva = lgb.Dataset(valid[feats], valid["trips"], reference=dtr)
mp = lgb.train(point_params, dtr, num_boost_round=2000, valid_sets=[dva],
               callbacks=[lgb.early_stopping(50), lgb.log_evaluation(0)])
pred_v = mp.predict(valid[feats], num_iteration=mp.best_iteration)
pred_t = mp.predict(test[feats],  num_iteration=mp.best_iteration)

# ---- 2. interval models (10% and 90%) ----
qbase = dict(learning_rate=0.1, num_leaves=63, min_data_in_leaf=100, feature_fraction=0.8,
             bagging_fraction=0.8, bagging_freq=1, seed=42, verbose=-1)
qv, qt = {}, {}
for name, alpha in {"lo": 0.1, "hi": 0.9}.items():
    mq = lgb.train({**qbase, "objective": "quantile", "alpha": alpha}, dtr, num_boost_round=500,
                   valid_sets=[dva], callbacks=[lgb.early_stopping(50), lgb.log_evaluation(0)])
    qv[name] = mq.predict(valid[feats], num_iteration=mq.best_iteration)
    qt[name] = mq.predict(test[feats],  num_iteration=mq.best_iteration)

def bounds(q):
    return np.clip(np.minimum(q["lo"], q["hi"]), 0, None), np.maximum(q["lo"], q["hi"])
lo_v, hi_v = bounds(qv)
lo_t, hi_t = bounds(qt)

# ---- 3. calibrate the intervals on ALL of July, separately by expected volume ----
yv = valid["trips"].values
edges = np.quantile(pred_v, [0.2, 0.4, 0.6, 0.8])
miss = np.maximum(lo_v - yv, yv - hi_v)
gv = np.digitize(pred_v, edges)
adj = np.zeros(5)
for gi in range(5):
    s = miss[gv == gi]
    n = len(s)
    adj[gi] = np.quantile(s, min(1.0, np.ceil((n + 1) * 0.8) / n))
a = adj[np.digitize(pred_t, edges)]
lo_c = np.clip(lo_t - a, 0, None)
hi_c = np.maximum(hi_t + a, lo_c)

# ---- 4. the August test ----
y = test["trips"].values
res = {"last value":          metrics(y, test[f"lag_{H}"]),
       "same hour yesterday": metrics(y, test["lag_24"]),
       "same hour last week": metrics(y, test["lag_168"]),
       "LightGBM (Poisson)":  metrics(y, pred_t)}
print(f"=== AUGUST TEST, horizon = {H} hour(s) ahead ===")
print(pd.DataFrame(res).T.round(2).to_string())

top20 = train.groupby("zone_id")["trips"].sum().nlargest(20).index
segs = {"all zones":       np.ones(len(test), bool),
        "airports":        (test["is_airport"] == 1).values,
        "top-20 zones":    test["zone_id"].isin(top20).values,
        "all other zones": ~test["zone_id"].isin(top20).values,
        "weekends":        (test["dayofweek"] >= 5).values}
inside_raw = (y >= lo_t) & (y <= hi_t)
inside_cal = (y >= lo_c) & (y <= hi_c)
seg_tab = pd.DataFrame({k: {"model WAPE %": wape(y[m], pred_t[m]),
                            "last-week WAPE %": wape(y[m], test["lag_168"].values[m]),
                            "interval coverage % (raw)": 100 * inside_raw[m].mean(),
                            "interval coverage % (calibrated)": 100 * inside_cal[m].mean()}
                        for k, m in segs.items()}).T.round(1)
print(seg_tab.to_string())
print("average interval width (trips): raw", round((hi_t - lo_t).mean(), 1),
      "| calibrated", round((hi_c - lo_c).mean(), 1), "| average demand:", round(y.mean(), 1))

final = globals().get("final", {})
final[H] = {"results": pd.DataFrame(res).T, "segments": seg_tab}

In [ ]:
def matched_share(df, forecast):
    t = pd.DataFrame({"hour": df["hour"].values, "y": df["trips"].values,
                      "fc": np.clip(np.asarray(forecast, float), 0, None)})
    supply = t.groupby("hour")["y"].transform("sum")                 # drivers available that hour
    fc_sum = t.groupby("hour")["fc"].transform("sum")
    alloc = np.where(fc_sum > 0, supply * t["fc"] / fc_sum, supply / t.groupby("hour")["y"].transform("size"))
    return 100 * np.minimum(t["y"], alloc).sum() / t["y"].sum()

sim = {"even split across zones":  matched_share(test, np.ones(len(test))),
       "same hour yesterday":      matched_share(test, test["lag_24"]),
       "same hour last week":      matched_share(test, test["lag_168"]),
       "LightGBM forecast":        matched_share(test, pred_t),
       "perfect forecast (limit)": matched_share(test, test["trips"])}
sim = pd.DataFrame({"trips with a driver in their zone %": sim})
sim["unmet %"] = 100 - sim.iloc[:, 0]
print(f"=== Allocation simulation, August, forecast made {H} hour(s) ahead ===")
print(sim.round(1).to_string())
base_unmet = sim.loc["same hour last week", "unmet %"]
model_unmet = sim.loc["LightGBM forecast", "unmet %"]
print(f"Unmet demand vs same-hour-last-week: {base_unmet:.1f}% -> {model_unmet:.1f}%  "
      f"({100 * (base_unmet - model_unmet) / base_unmet:.1f}% lower)")

In [ ]:
import os
from google.colab import drive
drive.mount('/content/drive')
assert os.path.isdir('/content/drive/MyDrive'), "Drive is not mounted"

out_dir = "/content/drive/MyDrive/uber_demand"
os.makedirs(out_dir, exist_ok=True)

fc = test[["hour", "zone_id", "Zone", "Borough", "trips"]].assign(
        forecast=pred_t, lo80=lo_c, hi80=hi_c, last_week=test["lag_168"].values)
fc.to_parquet(f"{out_dir}/august_forecasts_h{H}.parquet")
mp.save_model(f"{out_dir}/point_model_h{H}.txt")
print("saved for H =", H, "->", os.listdir(out_dir))